**APERÇU DE LA DATA**

In [26]:

import pandas as pd
import re, os
from datetime import datetime

RAW_PATH = r"C:\Users\delga\code\SAVIA\data\RAW\free tweet export 2.csv"

df_read = pd.read_csv(RAW_PATH)
df_read

,id,created_at,full_text,media,screen_name,name,profile_image_url,user_id,in_reply_to,retweeted_status,...,favorite_count,retweet_count,bookmark_count,quote_count,reply_count,views_count,favorited,retweeted,bookmarked,url
0,1343458257915031553,2020-12-28 08:26:23 +01:00,"💩 à @free parce-que Débit Très instable, … \n\...",[],m_annuel,M Annuel,https://abs.twimg.com/sticky/default_profile_i...,1104790986801250304,NaN,NaN,...,2,1,0,0,1,NaN,False,False,False,https://twitter.com/m_annuel/status/1343458257...
1,1393158240083587075,2021-05-14 12:56:22 +02:00,"RT @free: Retrouvez désormais @ToonamiFR, la c...","[{""type"":""photo"",""url"":""https://t.co/kuAYafYDi...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.393125e+18,...,0,16,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/13931582400...
2,1403337211475546112,2021-06-11 15:03:58 +02:00,"RT @free: A suivre ce soir, le 1er match de l’...","[{""type"":""photo"",""url"":""https://t.co/gMTcYtGdd...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403329e+18,...,0,15,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372114...
3,1403337257571004417,2021-06-11 15:04:09 +02:00,RT @free: Disponible sur le canal 101 avec les...,[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403333e+18,...,0,5,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372575...
4,1418550491034882052,2021-07-23 14:36:07 +02:00,« Faites vos premiers pas avec nous ! Découvre...,"[{""type"":""video"",""url"":""https://t.co/YCMv79evb...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,NaN,...,31,7,0,3,35,NaN,False,False,False,https://twitter.com/Freebox/status/14185504910...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6370,1934890606209196214,2025-06-17 10:27:23 +02:00,"Bonjour @free , c'est bien long pour trouver u...",[],JMA38620,Jean-Marc,https://pbs.twimg.com/profile_images/868547868...,802241352,NaN,NaN,...,0,0,0,0,2,115.0,False,False,False,https://twitter.com/JMA38620/status/1934890606...
6371,1934892363941302503,2025-06-17 10:34:22 +02:00,"@maarylou_v Bonjour,\n\nLa messagerie privée X...",[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,1.934651e+18,NaN,...,0,0,0,0,0,24.0,False,False,False,https://twitter.com/Freebox/status/19348923639...
6372,1934907898917691817,2025-06-17 11:36:06 +02:00,"@iac_yac Bonjour,\n\nLa messagerie privée X n’...",[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,1.934673e+18,NaN,...,0,0,0,0,0,30.0,False,False,False,https://twitter.com/Freebox/status/19349078989...
6373,1934909296245887334,2025-06-17 11:41:39 +02:00,"@MartinVine52691 Bonjour,\n\nLa messagerie pri...",[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,1.934690e+18,NaN,...,0,0,0,0,0,137.0,False,False,False,https://twitter.com/Freebox/status/19349092962...


**DATA CLEANING**

In [27]:



RAW_PATH = r"C:\Users\delga\code\SAVIA\data\RAW\free tweet export 2.csv"



timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")
print(f"{timestamp}")

SILVER_PATH = r"C:\Users\delga\code\SAVIA\data\silver\tweets_cleane_3.csv"
#SILVER_PATH = f"../data/silver/tweets_cleane_3.csv"
QUALITY_LOG_PATH  = "../quality/quality_log_tweets3.csv"



def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"@[\w_]+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", "", text, flags=re.UNICODE)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = pd.read_csv(RAW_PATH)
# 🧼 Application du nettoyage
df["clean_text"] = df["full_text"].apply(clean_text)
df["cleaned_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
     
# 🚫 Suppression des comptes officiels (Free, Iliad, Assistance, etc.)
EXCLUDED_ACCOUNTS = [
    "free", "free_1337", "free1337", "groupeiliad", "iliad",
    "free_officiel", "free_official", "groupe_iliad",
    "assistance freebox", "assistance_freebox"
]

# Normalisation des noms en minuscules sans accents (optionnel)
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
def text_length(s):
    if pd.isna(s):
        return 0
    return len(s)
df["screen_name_clean"] = df["screen_name"].apply(normalize_text)
df["name_clean"] = df["name"].apply(normalize_text)
df["text_length"] = df["clean_text"].apply(text_length)

# Filtrage
df_clean = df[
    ~df["screen_name_clean"].isin(EXCLUDED_ACCOUNTS) &
    ~df["name_clean"].isin(EXCLUDED_ACCOUNTS)&
    df["in_reply_to"].isna()
].copy()

print(f"[OK] {len(df) - len(df_clean)} comptes officiels exclus.")

df_clean[["screen_name", "name"]].drop_duplicates().head()
df_clean = df_clean[df_clean["text_length"] > 0]

os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
df_clean[["id", "created_at", "screen_name", "full_text", "clean_text", "cleaned_at"]].to_csv(SILVER_PATH, index=False)
print(f"[OK] Exporte vers : {SILVER_PATH}")

    

2025-11-23_14h29
[OK] 4099 comptes officiels exclus.
[OK] Exporte vers : C:\Users\delga\code\SAVIA\data\silver\tweets_cleane_3.csv


In [28]:
df[["full_text", "clean_text"]].head(10)

,full_text,clean_text
0,"💩 à @free parce-que Débit Très instable, … \n\...",à parceque débit très instable nnfree en franc...
1,"RT @free: Retrouvez désormais @ToonamiFR, la c...",rt retrouvez désormais la chaîne pour tous les...
2,"RT @free: A suivre ce soir, le 1er match de l’...",rt a suivre ce soir le 1er match de luefa euro...
3,RT @free: Disponible sur le canal 101 avec les...,rt disponible sur le canal 101 avec les freebo...
4,« Faites vos premiers pas avec nous ! Découvre...,faites vos premiers pas avec nous découvrez vo...
5,RT @free: ⚠️ Des personnes malveillantes se fo...,rt des personnes malveillantes se font passer ...
6,"RT @free: [NOUVEAU]\nAbonnés Freebox Pop, pour...",rt nouveaunabonnés freebox pop pour bien comme...
7,RT @BusyspiderFr: Astuce 💡Chez @Free les Email...,rt astuce chez les emails sont à vie mais suit...
8,FreePlugs et boîtiers CPL personnels : gérer v...,freeplugs et boîtiers cpl personnels gérer vot...
9,RT @free: Vous allez commencer le match avec 1...,rt vous allez commencer le match avec 15 min d...


**CLASSIFICATION**

In [29]:
import requests
import json
import pandas as pd
import os
# === CONFIGURATION ===
API_URL = "https://saviapi.win/classify"

MODEL = "Mistral-7B-Instruct"
PROJECT_ROOT = r"C:\Users\delga\code\SAVIA"

#PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

print(PROJECT_ROOT)
# Construction du chemin relatif
CSV_PATH = os.path.join(PROJECT_ROOT, "data", "silver", "tweets_cleaned_1.csv")

df = pd.read_csv(CSV_PATH)
tweet = df[df["id"]==1814368963826164172]["clean_text"].values[0]

CLASSIFICATION_CONTEXT = (
    "Tu es un assistant expert du Service Après-Vente (SAV) de l’opérateur Free. "
    "Ton objectif est d'analyser un tweet client et de déterminer s’il décrit un problème technique réel ou non, "
    "puis de classer ce problème par domaine et sous-domaine.\n\n"

    "Ne considère pas comme 'problème avéré' les tweets humoristiques, ironiques, publicitaires, "
    "ou ceux qui ne contiennent aucun signe de plainte réelle ou technique.\n\n"

    "Le résultat doit toujours être un JSON unique au format :\n"
    "{\"label\": label, \"domaine\": domaine, \"sous_domaine\": sous_domaine, \"score\": score}\n\n"

    "'score' = niveau de confiance entre 0.0 et 1.0.\n\n"

    "Choix possibles :\n"
    "- label : ['problème avéré', 'problème non avéré']\n"
    "- domaine : ['mobile', 'fixe', 'facture', 'contact']\n"
    "- sous_domaine : ['réseau', 'wifi', 'box', 'appel voix', 'sécurité', 'autres']\n\n"

    "Guide de décision :\n"
    "- Si le tweet demande un **conseiller, une aide, ou mentionne une panne/dysfonctionnement**, choisis 'problème avéré'.\n"
    "- Si le message est **vague, court, ironique ou sans signe de problème**, choisis 'problème non avéré'.\n"
    "- Si le texte parle de **connexion Internet, lenteur, perte de signal**, choisis domaine='fixe', sous_domaine='réseau' ou 'wifi'.\n"
    "- Si le texte parle de **carte SIM, 4G, 5G, appels, SMS**, choisis domaine='mobile', sous_domaine='appel voix' ou 'réseau'.\n"
    "- Si le texte évoque **facture, prélèvement, paiement, compte client**, choisis domaine='facture'.\n"
    "- Si le texte mentionne **mot de passe, piratage, sécurité**, choisis sous_domaine='sécurité'.\n"
    "- Si le tweet est un **mème, une blague, ou hors sujet technique**, choisis toujours 'problème non avéré'.\n\n"
    "- Si un tweet exprime une similarité avec un autre (par exemple “idem”, “pareil pour moi”, “moi aussi”), on privilégie alors la catégorie 'problème avéré'.\n"

    "Analyse le texte avec bon sens. Ne crée jamais de nouvelles catégories. "
    "Sois sobre, rigoureux et évite toute surclassification."
)

payload = {
        "prompt": tweet,
        "model": MODEL,
        "context": CLASSIFICATION_CONTEXT
    }

print(f" Envoi du texte à classifier vers {API_URL}...")
print(f"Contenu : {tweet}\n")

try:
    response = requests.post(API_URL, json=payload, timeout=60)
    print(f"Status code : {response.status_code}")
    print(" Réponse brute :")
    print(response.text)

    # Si la réponse est au format JSON
    try:
        data = response.json()
        print("\n JSON parsé :")
        print(json.dumps(data, indent=2, ensure_ascii=False))
    except Exception:
        print("\n Impossible de parser la réponse en JSON")

except requests.exceptions.RequestException as e:
    print(f" Erreur de connexion : {e}")


C:\Users\delga\code\SAVIA
 Envoi du texte à classifier vers https://saviapi.win/classify...
Contenu : idem 93

Status code : 200
 Réponse brute :
{"label":"problème avéré","domaine":"fixe","sous_domaine":"réseau","score":0.9,"text":"idem 93"}

 JSON parsé :
{
  "label": "problème avéré",
  "domaine": "fixe",
  "sous_domaine": "réseau",
  "score": 0.9,
  "text": "idem 93"
}
